In [1]:
using Pkg

In [2]:
Pkg.activate(".")

  Activating project at `~/Documents/GitHub/PhDThesis/implementations/TDA`


In [3]:
using Oscar
using CGPlots
using NetworkLayout
# using Graphs
import Combinatorics: powerset
using OrderedCollections
import Oscar: vertices, faces, edges, connected_components

  ___   ____   ____    _    ____
 / _ \ / ___| / ___|  / \  |  _ \   |  Combining ANTIC, GAP, Polymake, Singular
| | | |\___ \| |     / _ \ | |_) |  |  Type "?Oscar" for more information
| |_| | ___) | |___ / ___ \|  _ <   |  Manual: https://docs.oscar-system.org
 \___/ |____/ \____/_/   \_\_| \_\  |  Version 1.0.3


[ Info: Precompiling CGPlots [8c229dad-8b3a-4031-83d6-73545c88426d]


In [4]:
powerset(k::SimplicialComplex) = OrderedSet(mapreduce(collect∘powerset∘ x->1 .+ collect(x),vcat,k.pm_simplicialcomplex.FACETS,init=[]))
labels(k::SimplicialComplex) = map(x->parse(Int,x),collect(k.pm_simplicialcomplex.VERTEX_LABELS))
vertices(k::SimplicialComplex) = vertexindices(k)

vectorize_orderedset(o) = mapreduce(x->[x], vcat, o,init=[])
edges(k::SimplicialComplex) = vectorize_orderedset(filter(x->length(x) == 2, powerset(k)))
faces(k::SimplicialComplex) = vectorize_orderedset(filter(x->length(x) == 3, powerset(k)))
subcomplex(k::SimplicialComplex,n::Int) = vectorize_orderedset(filter(x->length(x) == n, powerset(k)))

# connected_components(K::SimplicialComplex) = length(facets(K))

subcomplex (generic function with 1 method)

In [5]:
K = simplicial_complex([[1],[2],[3],[1,2],[2,3],[1,3],[1,2,3]])
euler_characteristic(K)

0

In [6]:
@show subcomplex(K,1)
@show subcomplex(K,2)
@show subcomplex(K,3);

subcomplex(K, 1) = Any[[1], [2], [3]]
subcomplex(K, 2) = Any[[1, 2], [1, 3], [2, 3]]
subcomplex(K, 3) = Any[[1, 2, 3]]


In [18]:
s = powerset(K)

OrderedSet{Any} with 4 elements:
  Int64[]
  [1]
  [2]
  [3]

In [19]:
k = simplicial_complex([[1,2,3,4]])
k = simplicial_complex([[1],[2],[3],[1,2],[2,3],[1,3]])

Abstract simplicial complex of dimension 1 on 3 vertices

In [65]:
function compute_adjacency_matrix(vertices::Vector, edges::Vector)
    n = length(vertices)
    vertex_to_index = Dict(v => i for (i, v) in enumerate(vertices))
    adj_matrix = zeros(Int, n, n)
    
    for (u, v) in edges
        i, j = vertex_to_index[u], vertex_to_index[v]
        adj_matrix[i, j] = 1
        adj_matrix[j, i] = 1  # Remove this line if the graph is directed
    end
    
    return adj_matrix
end

compute_adjacency_matrix (generic function with 1 method)

In [70]:
function draw(K::SimplicialComplex)
    pos = shell(compute_adjacency_matrix(vertices(K),edges(K)))
    r = mapreduce(x->norm(x-pos[1]),max,pos)/10
    
    v = mapreduce(x->T(x...)Circle(r=0.1),+, pos)
    
    v = S(:fill=>:grey)mapreduce(x->begin 
            T(x[1]...)*(Circle(r=r)+U(r)S(:fill=>:white)TextMark(text=x[2],fontsize=0.8))
        end
            ,+, zip(pos,vertices(K)),init=NilD())

    e = mapreduce(x->Line(pos[x]),+,edges(K),init=NilD())
    f = S(:fillOpacity=>0.1)mapreduce(x->CGPlots.Polygon(pos[x]),+,faces(K),init=NilD())
    
    return drawsvg(f+e+v)
end

K = simplicial_complex([[1],[2],[3],[1,2],[2,3],[1,3],[1,2,3]])
draw(K)

<svg height="310" viewBox="0 0 276.50635094610965 310" width="276.50635094610965" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="246.5063509461097 155.00000000000003 30.0000000000001 30.0 30.0 280.0 " style="fill-opacity: 0.1; " /><polyline points="246.5063509461097 155.00000000000003 30.0000000000001 30.0 " style="stroke: black; fill: none; " /><polyline points="246.5063509461097 155.00000000000003 30.0 280.0 " style="stroke: black; fill: none; " /><polyline points="30.0000000000001 30.0 30.0 280.0 " style="stroke: black; fill: none; " /><circle cy="155.00000000000003" style="fill: grey; " r="25.0" cx="246.5063509461097" /><text x="246.5063509461097" style="font-family: Helvetica; text-anchor: middle; font-size: 20.00000000000003pt; fill: white; " transform="rotate(-0.0, 246.5063509461097, 165.00000000000003)" y="165.00000000000003">1</text><circle cy="30.0" style="fill: grey; " r="25.000000000000004" cx="30.000000000000103" /><text x="30.0000000000001" style="font-family: Helvetica; text-anchor: middle; font-size: 20.0pt; fill: white; " transform="rotate(-0.0, 30.0000000000001, 40.0)" y="40.0">2</text><circle cy="280.0" style="fill: grey; " r="25.000000000000007" cx="30.000000000000007" /><text x="30.00000000000001" style="font-family: Helvetica; text-anchor: middle; font-size: 19.999999999999993pt; fill: white; " transform="rotate(-0.0, 30.00000000000001, 290.0)" y="290.0">3</text></svg>

In [71]:
K = simplicial_complex([[4,5],[1,2,3]])


draw(K)

<svg height="310" viewBox="0 0 297.76412907378847 310" width="297.76412907378847" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="267.76412907378847 154.99999999999994 176.94631307311835 29.999999999999886 30.00000000000003 77.74575140626305 " style="fill-opacity: 0.1; " /><polyline points="30.000000000000014 232.2542485937368 176.9463130731183 280.0 " style="stroke: black; fill: none; " /><polyline points="267.76412907378847 154.99999999999994 176.94631307311835 29.999999999999886 " style="stroke: black; fill: none; " /><polyline points="267.76412907378847 154.99999999999994 30.00000000000003 77.74575140626305 " style="stroke: black; fill: none; " /><polyline points="176.94631307311835 29.999999999999886 30.00000000000003 77.74575140626305 " style="stroke: black; fill: none; " /><circle cy="154.99999999999994" style="fill: grey; " r="25.0" cx="267.76412907378847" /><text x="267.76412907378847" style="font-family: Helvetica; text-anchor: middle; font-size: 20.0pt; fill: white; " transform="rotate(-0.0, 267.76412907378847, 164.99999999999994)" y="164.99999999999994">1</text><circle cy="29.999999999999886" style="fill: grey; " r="25.0" cx="176.94631307311835" /><text x="176.94631307311835" style="font-family: Helvetica; text-anchor: middle; font-size: 20.00000000000003pt; fill: white; " transform="rotate(-0.0, 176.94631307311835, 39.99999999999994)" y="39.99999999999994">2</text><circle cy="77.74575140626305" style="fill: grey; " r="25.0" cx="30.00000000000003" /><text x="30.00000000000002" style="font-family: Helvetica; text-anchor: middle; font-size: 20.000000000000014pt; fill: white; " transform="rotate(-0.0, 30.00000000000002, 87.74575140626305)" y="87.74575140626305">3</text><circle cy="232.2542485937368" style="fill: grey; " r="25.000000000000007" cx="30.000000000000007" /><text x="30.000000000000007" style="font-family: Helvetica; text-anchor: middle; font-size: 20.000000000000014pt; fill: white; " transform="rotate(-0.0, 30.000000000000007, 242.2542485937368)" y="242.2542485937368">4</text><circle cy="280.0" style="fill: grey; " r="24.999999999999986" cx="176.9463130731183" /><text x="176.9463130731183" style="font-family: Helvetica; text-anchor: middle; font-size: 20.00000000000003pt; fill: white; " transform="rotate(-0.0, 176.9463130731183, 290.0)" y="290.0">5</text></svg>

In [72]:
K = simplicial_complex([[4,1],[1,2,3]])

draw(K)

<svg height="310" viewBox="0 0 310.0 310" width="310.0" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="280.0 155.0 155.0 30.0 30.0 154.99999999999997 " style="fill-opacity: 0.1; " /><polyline points="280.0 155.0 154.99999999999997 280.0 " style="stroke: black; fill: none; " /><polyline points="280.0 155.0 155.0 30.0 " style="stroke: black; fill: none; " /><polyline points="280.0 155.0 30.0 154.99999999999997 " style="stroke: black; fill: none; " /><polyline points="155.0 30.0 30.0 154.99999999999997 " style="stroke: black; fill: none; " /><circle cy="155.0" style="fill: grey; " r="25.0" cx="280.0" /><text x="280.0" style="font-family: Helvetica; text-anchor: middle; font-size: 20.0pt; fill: white; " transform="rotate(-0.0, 280.0, 165.0)" y="165.0">1</text><circle cy="30.0" style="fill: grey; " r="25.0" cx="155.0" /><text x="155.0" style="font-family: Helvetica; text-anchor: middle; font-size: 20.0pt; fill: white; " transform="rotate(-0.0, 155.0, 40.0)" y="40.0">2</text><circle cy="154.99999999999997" style="fill: grey; " r="25.0" cx="30.0" /><text x="30.000000000000007" style="font-family: Helvetica; text-anchor: middle; font-size: 20.000000000000014pt; fill: white; " transform="rotate(-0.0, 30.000000000000007, 165.0)" y="165.0">3</text><circle cy="280.0" style="fill: grey; " r="25.0" cx="154.99999999999997" /><text x="154.99999999999997" style="font-family: Helvetica; text-anchor: middle; font-size: 20.0pt; fill: white; " transform="rotate(-0.0, 154.99999999999997, 290.0)" y="290.0">4</text></svg>

In [73]:
K = simplicial_complex([[1],[2],[3],[1,2],[2,3],[1,3]])
draw(K)

betti_numbers(K)

2-element Vector{Int64}:
 0
 1

In [74]:
K = simplicial_complex([[1],[2],[3]])

draw(K)

<svg height="310" viewBox="0 0 276.50635094610965 310" width="276.50635094610965" xmlns="http://www.w3.org/2000/svg" version="1.1"><circle cy="155.00000000000003" style="fill: grey; " r="25.0" cx="246.5063509461097" /><text x="246.5063509461097" style="font-family: Helvetica; text-anchor: middle; font-size: 20.00000000000003pt; fill: white; " transform="rotate(-0.0, 246.5063509461097, 165.00000000000003)" y="165.00000000000003">1</text><circle cy="30.0" style="fill: grey; " r="25.000000000000004" cx="30.000000000000103" /><text x="30.0000000000001" style="font-family: Helvetica; text-anchor: middle; font-size: 20.0pt; fill: white; " transform="rotate(-0.0, 30.0000000000001, 40.0)" y="40.0">2</text><circle cy="280.0" style="fill: grey; " r="25.000000000000007" cx="30.000000000000007" /><text x="30.00000000000001" style="font-family: Helvetica; text-anchor: middle; font-size: 19.999999999999993pt; fill: white; " transform="rotate(-0.0, 30.00000000000001, 290.0)" y="290.0">3</text></svg>

In [76]:
# methodswith(SimplicialComplex, supertypes=true)

In [17]:
# K.pm_simplicialcomplex.BOUNDARY

In [80]:
K.pm_simplicialcomplex
fundamental_group(K)

polymake:  WARNING: could not compute 'FUNDAMENTAL_GROUP' probably because of unsatisfied preconditions:
precondition : GRAPH.CONNECTED ( FUNDAMENTAL_GROUP, FUNDAMENTAL_GROUP_GEN_LABELS : FACETS, GRAPH.ADJACENCY )


LoadError: MethodError: no method matching iterate(::Nothing)

[0mClosest candidates are:
[0m  iterate([91m::Base.MethodSpecializations[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mreflection.jl:1148[24m[39m
[0m  iterate([91m::Base.MethodSpecializations[39m, [91m::Nothing[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mreflection.jl:1154[24m[39m
[0m  iterate([91m::Base.MethodSpecializations[39m, [91m::Int64[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mreflection.jl:1155[24m[39m
[0m  ...
